In [ ]:
!pip install -q pypdf

In [125]:
import google.generativeai as genai
from google.colab import userdata
import numpy as np

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

In [126]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("uploaded", pdf_name)

Saving Sunridge Institute of Technology — Student Policy Handbook 2025–26.pdf to Sunridge Institute of Technology — Student Policy Handbook 2025–26 (2).pdf
uploaded Sunridge Institute of Technology — Student Policy Handbook 2025–26 (2).pdf


In [127]:
from pypdf import PdfReader
reader = PdfReader(pdf_name)
print("no of pages:", len(reader.pages))

text = ""
for page in reader.pages:
  text += page.extract_text() + "\n"

print("total characters:", len(text))
print(text[:801])


no of pages: 6
total characters: 11210
Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Technology, Devgarh
Campus. It covers admissions, the refund of fees, hostel residence, examinations and
re-evaluation, library usage, attendance, scholarships and the student grievance
procedure. In the event of any dispute, the version of this handbook published on the
Registrar's notice board on 1 July 2025 shall be treated as final. Students are expected
to read this handbook in full within the first two weeks of the semester.
1. Admission Rules
Admission to all undergraduate programmes at Sunridge Institute of Technology 


In [128]:
 #chunk0 : 1 to 800, chunk1: 650 to 1450
def chunk_text(text, chunk_size = 800, overlap = 150):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start = end - overlap
  return chunks

chunks = chunk_text(text)
print(len(chunks))


18


In [129]:
print("Number of chunks:", len(chunks))

print("\nSample Chunk:\n")
print(chunks[0])

Number of chunks: 18

Sample Chunk:

Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Technology, Devgarh
Campus. It covers admissions, the refund of fees, hostel residence, examinations and
re-evaluation, library usage, attendance, scholarships and the student grievance
procedure. In the event of any dispute, the version of this handbook published on the
Registrar's notice board on 1 July 2025 shall be treated as final. Students are expected
to read this handbook in full within the first two weeks of the semester.
1. Admission Rules
Admission to all undergraduate programmes at Sunridge Institute of Technology


In [130]:
#Embedding the chunks
def embed_text(t):
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=t,
        output_dimensionality=768
    )
    return np.array(result["embedding"])

chunk_embeddings = []
for i, ch in enumerate(chunks):
    print(i, end=" ")
    chunk_embeddings.append(embed_text(ch))
    print(f"embeed chunk {i+1}/{len(chunks)}", end="\r")
chunk_embeddings = np.array(chunk_embeddings)
print("shape:", chunk_embeddings.shape)


shape: (18, 768)


In [131]:
chunk_embeddings = chunk_embeddings /np.linalg.norm(
    chunk_embeddings, axis=1, keepdims=True
)

print("store", len(chunks), "chunks + ", chunk_embeddings.shape)

store 18 chunks +  (18, 768)


In [132]:
import numpy as np
# question -- search --- top-k chunks

def retrieve(question, k=3):
  question_embedding = embed_text(question)
  question_embedding = question_embedding / np.linalg.norm(question_embedding)
  scores = chunk_embeddings @ question_embedding    # shape: (6,)
  top_indices = np.argsort(scores)[::-1][:k]

  return [(i, chunks[i], scores[i]) for i in top_indices]

In [133]:
results = retrieve("How do i get my fees back i cancel admissing?")


for i,chunk, score in results:
  print(f"score:{score:.3f}")
  print(chunk)
  print()

score:0.701
e are not treated
as valid cancellation requests under any circumstances.
If the cancellation request is received before the official commencement of
classes, the institute will process a refund of fees within 12 working days of
receiving the request, after deducting 10 percent of the total fees paid as
administrative charges. The admission confirmation fee of ₹18,500 is fully non-
refundable in all cases.
If the cancellation request is received within 15 days after the commencement of
classes, the deduction rises to 25 percent of the total fees paid, and the refund is
processed within 20 working days. If the request is received more than 15 days after
the commencement of classes, no refund of tuition fees is payable; only the
caution deposit of ₹4,000 is returned.
All refunds are made exc

score:0.677
ed at 10 percent of the sanctioned strength of each programme.
Change of branch is permitted only once, at the end of the first year, and only for
students with a CGPA of 8.6

In [134]:
llm = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    system_instruction=(
      "you answer question using only the context provided to you."
      "if the answer is not in the context, reply exactly: "
      "'I could not find this in the document.' Never guess."
    )
)

def ask(question, k=3):
    top = retrieve(question, k)

    context = "\n\n---\n".join(chunk for _, chunk, _ in top)

    prompt = f"""Context from the document:
    {context}

    QUESTION: {question}

    Answer using only the context above."""

    response = llm.generate_content(prompt)

    answer = response.text.strip()

    print(answer)

    if answer != "I could not find this in the document.":
        print(f"\nSimilarity Score: {top[0][2]:.3f}")
        print("\nFrom the chunk:\n")
        print(top[0][1])

## Test 1: Questions Present in the PDF

In [135]:
ask("What is the fee refund policy for cancelled admission?")

Based on the provided context, the fee refund policy for cancelled admission is as follows:

* **Cancellation request received before the official commencement of classes:** The institute processes a refund within 12 working days of receiving the request, after deducting 10 percent of the total fees paid as administrative charges. The admission confirmation fee of ₹18,500 is fully non-refundable in all cases.
* **Cancellation request received within 15 days after the commencement of classes:** The deduction rises to 25 percent of the total fees paid, and the refund is processed within 20 working days.
* **Cancellation request received more than 15 days after the commencement of classes:** No refund of tuition fees is payable; only the caution deposit of ₹4,000 is returned.

**Additional conditions:**
* A written cancellation request must be submitted on Form CR-2 to the Office of the Registrar, along with the original fee receipt. Requests sent by email or made over the telephone are n

In [136]:
ask("What is the annual hostel fee for a triple sharing room?")

The annual hostel fee for a triple-sharing room is ₹52,000.

Similarity Score: 0.703

From the chunk:

 be entertained.
3. Hostel Rules
The institute operates three hostels: Aravalli House and Nilgiri House for male
students, and Shivalik House for female students. Allotment of hostel rooms is made
strictly in order of distance of the student's home town from the campus, with students
from beyond 300 kilometres given first preference. The annual hostel fee is ₹52,000
for a triple-sharing room and ₹78,000 for a double-sharing room, payable in two equal
instalments in July and December.
The hostel gates close at 9:45 PM on all days, including weekends and holidays. A
resident returning after 9:45 PM must sign the late register at the warden's office; three
late entries in a single month lead to an automatic fine of ₹600 and a written warning
copied to the student's parents. Overnight absence 


In [137]:
ask("What is the minimum attendance required to appear for the end semester examination?")

A minimum attendance of 75 percent in each course is mandatory for appearing in the end-semester examination of that course.

Similarity Score: 0.727

From the chunk:

t issued for
borrowing and may be consulted only inside the reading hall. Marking, underlining or
folding pages of library books is treated as damage and fined at ₹300 per book. The
library conducts its annual stock verification in the last week of April, during which all
borrowed books must be returned regardless of due dates.
6. Attendance and Leave
A minimum attendance of 75 percent in each course is mandatory for appearing in the
end-semester examination of that course. Attendance is computed course-wise, not as
an aggregate. A student whose attendance in a course falls between 65 and 75 percent
may apply for condonation on medical grounds only, supported by a medical
certificate, on payment of a condonation fee of ₹800 per course; condonation below 65
percent is not granted by any aut


## Test 2: Questions Not Present in the PDF

In [ ]:
ask("Who is the Prime Minister of India?")

I could not find this in the document.


In [ ]:
ask("What is the placement percentage of the college?")

I could not find this in the document.


In [ ]:
ask("Who is the Principal of Sunridge Institute?")

I could not find this in the document.


Tests observation
- The chatbot answered all questions that were present in the PDF correctly.
- The retrieved chunks contained the required information for generating the answers.
- For questions not present in the PDF, the chatbot correctly responded that it could not find the information from the uploaded pdf or document.


## Experiment (Chunk Size = 3000)

In [ ]:
chunks = chunk_text(text, chunk_size=3000, overlap=150)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [ ]:
chunk_embeddings = []

for i, ch in enumerate(chunks):
    print(i, end=" ")
    chunk_embeddings.append(embed_text(ch))
    print(f"embed chunk {i+1}/{len(chunks)}", end="\r")

chunk_embeddings = np.array(chunk_embeddings)

print("shape:", chunk_embeddings.shape)

shape: (4, 768)


In [ ]:
chunk_embeddings = chunk_embeddings / np.linalg.norm(
    chunk_embeddings,
    axis=1,
    keepdims=True
)

print("store", len(chunks), "chunks +", chunk_embeddings.shape)

store 4 chunks + (4, 768)


In [ ]:
ask("What is the fee refund policy for cancelled admission?")

Based on the provided document, the fee refund policy for a cancelled admission is as follows:

* **Cancellation request before the official commencement of classes:** The institute will process a refund of fees within 12 working days of receiving the request, after deducting 10 percent of the total fees paid as administrative charges.
* **Cancellation request within 15 days after the commencement of classes:** The deduction rises to 25 percent of the total fees paid, and the refund is processed within 20 working days.
* **Cancellation request received more than 15 days after the commencement of classes:** No refund of tuition fees is payable; only the caution deposit of ₹4,000 is returned.
* **Admission confirmation fee:** The confirmation fee of ₹18,500 is fully non-refundable in all cases.
* **Method of refund:** All refunds are made exclusively by electronic transfer to the bank account named in the original admission form (never paid in cash or by cheque).
* **Delayed refunds:** I

In [ ]:
ask("What is the annual hostel fee for a triple sharing room?")

The annual hostel fee for a triple-sharing room is ₹52,000.

Similarity Score: 0.664

From the chunk:

, after deducting 10 percent of the total fees paid as
administrative charges. The admission confirmation fee of ₹18,500 is fully non-
refundable in all cases.
If the cancellation request is received within 15 days after the commencement of
classes, the deduction rises to 25 percent of the total fees paid, and the refund is
processed within 20 working days. If the request is received more than 15 days after
the commencement of classes, no refund of tuition fees is payable; only the
caution deposit of ₹4,000 is returned.
All refunds are made exclusively by electronic transfer to the bank account named in
the original admission form. Refunds are never paid in cash or by cheque. If a refund is
delayed beyond the stated period through the fault of the institute, the student is
entitled to simple interest at 6 percent per annum on the refundable amount,
calculated from the day the refund f

In [ ]:
ask("What is the minimum attendance required to appear for the end semester examination?")

To appear for the end-semester examination, a student must have a minimum attendance of 75 percent in that course.

Similarity Score: 0.719

From the chunk:

nations and Re-evaluation Procedure
End-semester examinations are held twice a year, in December and in May. To be
eligible to sit an end-semester examination, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues. A student
who misses an examination on medical grounds may apply for a make-up examination
within 10 days, attaching a medical certificate issued by a registered practitioner;
make-up examinations are held in the first week of the following month.
A student who is dissatisfied with the marks awarded in any theory paper may apply for
re-evaluation. The application must be made online through the examination portal
within 14 days of the declaration of results, and the re-evaluation fee is ₹460 per
paper. Re-evaluation is not available for laboratory courses, project work,

Experiment Observation

- Increasing the chunk size from 800 to 3000 reduced the total number of chunks from 18 to 4.
- The similarity scores are increased, and the chatbot still produced the correct answers for all three questions.
- But the retrieved chunks became much larger and contained additional information that was not directly related to the question. For this document, the larger chunk size improved the similarity score, but the smaller chunk size of 800 provided more focused context.